# Day 3 - Part 2: 자연어 처리 기초 - 실습 과제

이번 과제에서는 Day 3 Part 2 튜토리얼에서 배운 내용을 바탕으로 `한국어 멀티 클래스 감정 분류` 모델을 직접 구현하고 학습시켜 봅니다.

`과제 목표:`

1. 튜토리얼에서 제시된 전처리 과정을 이해하고, 주어진 코드의TODO부분을 완성하여 전체 전처리 파이프라인을 구축합니다.

2. torch.nn모듈을 사용하여 멀티 클래스 감정 분류를 위한 LSTM 기반 모델을 직접 설계하고 구현합니다.
3. 모델을 학습시키고, 테스트 데이터에 대한 성능(정확도)을 측정합니다.
4. 임의의 새로운 문장에 대해 학습된 모델이 감정 예측을 수행할 수 있도록 predict_emotion함수를 완성합니다.
5. 하이퍼파라미터 튜닝, 다른 RNN 셀(GRU 등) 적용, 또는 추가적인 전처리 기법 적용 등을 통해 모델 성능을 향상시켜 봅니다.

## 1. 데이터 준비 및 전처리

In [25]:
import pandas as pd
df = pd.read_excel('../../datasets/dl/ko-onetime-sentiment/ko_onetime_sentiment.xlsx', usecols=["Sentence", "Emotion"])
df.head()

,Sentence,Emotion
0,언니 동생으로 부르는게 맞는 일인가요..??,공포
1,그냥 내 느낌일뿐겠지?,공포
2,아직너무초기라서 그런거죠?,공포
3,유치원버스 사고 낫다던데,공포
4,근데 원래이런거맞나요,공포


In [26]:
df.Emotion.value_counts()

Emotion
행복    6037
놀람    5898
분노    5665
공포    5468
혐오    5429
슬픔    5267
중립    4830
Name: count, dtype: int64

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

# 한국어 감정 분류 데이터 로드
df = pd.read_excel('../../datasets/dl/ko-onetime-sentiment/ko_onetime_sentiment.xlsx', usecols=["Sentence", "Emotion"])

# 감정 레이블을 숫자로 변환
emotion_to_idx = {
    '행복': 0, '놀람': 1, '분노': 2, '공포': 3, 
    '혐오': 4, '슬픔': 5, '중립': 6
}
df['emotion_idx'] = df['Emotion'].map(emotion_to_idx)

# 한국어 텍스트 정제 함수
def preprocess_korean_text(text):
    # 특수문자 제거 (한글, 영문, 숫자, 공백만 유지)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)
    # 연속된 공백을 하나로 치환
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 텍스트 전처리 적용
df['cleaned_sentence'] = df['Sentence'].apply(preprocess_korean_text)
tokenized_texts = [sentence.split() for sentence in df['cleaned_sentence']]

# 단어 사전 구축 (빈도수 기준 상위 vocab_size - 1 개 단어 사용, <unk> 토큰 포함)
vocab_size = 5000  # 한국어 데이터에 맞게 조정
word_counts = Counter(word for tokens in tokenized_texts for word in tokens)
vocab = [word for word, count in word_counts.most_common(vocab_size - 1)]
word_to_idx = {word: idx + 1 for idx, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0

# 정수 인코딩 (사전에 없는 단어는 <unk> 토큰으로 처리)
def encode_text(tokens):
    return [word_to_idx.get(word, 0) for word in tokens]

encoded_sequences = [encode_text(tokens) for tokens in tokenized_texts]

# 패딩 (한국어 문장 길이에 맞게 조정)
max_len = 50  # 한국어 문장은 보통 더 짧음
padded_sequences = np.array([
    seq[:max_len] + [0] * (max_len - len(seq)) if len(seq) < max_len 
    else seq[:max_len] for seq in encoded_sequences
])

# 훈련/테스트 데이터 분리
X = padded_sequences
y = df['emotion_idx'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# PyTorch 텐서 변환 및 DataLoader 생성
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.LongTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.LongTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

batch_size = 32  # 멀티클래스 분류에 맞게 조정

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("한국어 감정 분류 데이터 준비 및 전처리 완료!")
print(f"훈련 데이터 크기: {len(X_train)}")
print(f"테스트 데이터 크기: {len(X_test)}")
print(f"감정 클래스 수: {len(emotion_to_idx)}")
print(f"어휘 크기: {vocab_size}")

한국어 감정 분류 데이터 준비 및 전처리 완료!
훈련 데이터 크기: 30875
테스트 데이터 크기: 7719
감정 클래스 수: 7
어휘 크기: 5000


## 2. 감성 분석 모델 정의

In [28]:
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        # 임베딩 레이어 정의 (vocab_size, embedding_dim, padding_idx=0)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # LSTM 레이어 정의 (embedding_dim, hidden_dim, n_layers, dropout, batch_first=True)
        self.lstm = nn.LSTM(embedding_dim, 
                           hidden_dim, 
                           n_layers, 
                           dropout=drop_prob, 
                           batch_first=True)
        
        # 드롭아웃 레이어 정의 (drop_prob)
        self.dropout = nn.Dropout(drop_prob)
        
        # 완전 연결 레이어 정의 (hidden_dim, output_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # 임베딩 레이어 통과
        embedded = self.embedding(x) # (batch_size, seq_length, embedding_dim)
        
        # LSTM 레이어 통과 (hidden state만 필요)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # 마지막 hidden state 추출 (n_layers, batch_size, hidden_dim) -> (batch_size, hidden_dim)
        last_hidden = hidden[-1]
        
        # 드롭아웃 적용
        out = self.dropout(last_hidden)
        
        # 완전 연결 레이어 통과
        out = self.fc(out)
        
        return out

# 모델 하이퍼파라미터 설정
embedding_dim = 128
hidden_dim = 256
output_dim = len(emotion_to_idx)  # 감정 클래스 수에 맞게 수정
n_layers = 2

# 모델 인스턴스 생성
model = SentimentLSTM(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers)
print(model)

SentimentLSTM(
  (embedding): Embedding(5000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=7, bias=True)
)


## 3. 모델 학습 및 평가

In [29]:
import torch.optim as optim
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 손실 함수 및 옵티마이저 정의
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 설정
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 학습 과정 추적을 위한 리스트
train_losses = []
test_accuracies = []

# 학습 루프
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # 평가
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    test_accuracies.append(accuracy)
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Accuracy: {accuracy:.2f}%')

print("Training finished!")

# 학습 과정 시각화
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Training Loss', 'Test Accuracy'))

fig.add_trace(
    go.Scatter(y=train_losses, mode='lines+markers', name='Train Loss'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=test_accuracies, mode='lines+markers', name='Test Accuracy'),
    row=1, col=2
)

fig.update_layout(
    title='감정 분류 모델 학습 과정',
    height=400,
    showlegend=True
)

fig.show()

Epoch [1/10], Train Loss: 1.9449, Test Accuracy: 15.64%
Epoch [2/10], Train Loss: 1.9445, Test Accuracy: 15.29%
Epoch [3/10], Train Loss: 1.9445, Test Accuracy: 14.68%
Epoch [4/10], Train Loss: 1.9442, Test Accuracy: 15.64%
Epoch [5/10], Train Loss: 1.9446, Test Accuracy: 15.64%
Epoch [6/10], Train Loss: 1.9445, Test Accuracy: 15.64%
Epoch [7/10], Train Loss: 1.9442, Test Accuracy: 15.64%
Epoch [8/10], Train Loss: 1.9441, Test Accuracy: 15.64%
Epoch [9/10], Train Loss: 1.9437, Test Accuracy: 15.64%
Epoch [10/10], Train Loss: 1.9436, Test Accuracy: 15.64%
Training finished!


## 4. 새로운 문장 감성 예측 함수

In [30]:
def predict_emotion(text):
    model.eval()
    
    # 1. 텍스트 전처리
    cleaned_text = preprocess_korean_text(text)
    tokenized = cleaned_text.split()
    
    # 2. 정수 인코딩 (사전에 없는 단어는 <unk>)
    encoded = [word_to_idx.get(word, 0) for word in tokenized]
    
    # 3. 패딩
    padded = np.array([encoded[:max_len] + [0]*(max_len - len(encoded)) if len(encoded) < max_len else encoded[:max_len]])
    
    # 4. 텐서 변환 및 장치 할당
    # (batch_size=1, seq_len) 형태로 변환
    input_tensor = torch.LongTensor(padded).to(device)  # shape: (1, seq_len)
    
    # 5. 예측
    with torch.no_grad():
        output = model(input_tensor)
        _, prediction = torch.max(output, 1)
    
    # 감정 레이블 매핑
    emotion_labels = ['행복', '놀람', '분노', '공포', '혐오', '슬픔', '중립']
    predicted_emotion = emotion_labels[prediction.item()]
    
    return predicted_emotion

# 테스트
test_sentence_1 = "오늘 정말 기분이 좋아요! 새로운 일이 생겨서 너무 신나요."
test_sentence_2 = "이런 상황이 정말 화가 나네요. 도저히 참을 수 없어요."

print(f"문장 1: '{test_sentence_1}' -> 예측 감정: {predict_emotion(test_sentence_1)}")
print(f"문장 2: '{test_sentence_2}' -> 예측 감정: {predict_emotion(test_sentence_2)}")


문장 1: '오늘 정말 기분이 좋아요! 새로운 일이 생겨서 너무 신나요.' -> 예측 감정: 행복
문장 2: '이런 상황이 정말 화가 나네요. 도저히 참을 수 없어요.' -> 예측 감정: 행복


## 5. 모델 성능 향상시키기

다음과 같은 방법들을 시도하여 모델의 성능을 향상시켜 보세요.

* `하이퍼파라미터 튜닝:`embedding_dim,hidden_dim,n_layers,dropout등의 값을 다양하게 변경하여 학습시켜 보세요.

* `다른 RNN 셀 사용:`nn.LSTM대신nn.GRU를 사용하여 모델을 재구축하고 학습시켜 보세요.
* `전처리 개선:` 불용어(stopwords) 제거, 어간(stemming) 또는 표제어 추출(lemmatization) 등의 추가적인 전처리 기법을 적용해보세요.
* `더 많은 데이터 또는 사전 훈련된 임베딩 사용:` GloVe나 Word2Vec과 같은 사전 훈련된 워드 임베딩을nn.Embedding레이어에 로드하여 사용해보세요.
* `모델 구조 변경:` 양방향 LSTM(Bidirectional LSTM) 레이어를 적용해보세요.
* `더 많은 데이터:` 어떻게 데이터를 추가할수 있을까요? ChatGPT 제로샷?

자유롭게 실험하고 결과를 분석하여 더 좋은 성능을 내는 모델을 만들어 보세요!

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ImprovedSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, drop_prob=0.3):
        super(ImprovedSentimentLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # 임베딩 레이어 (더 큰 임베딩 차원)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 양방향 LSTM 레이어
        self.lstm = nn.LSTM(embedding_dim, 
                           hidden_dim, 
                           n_layers, 
                           dropout=drop_prob if n_layers > 1 else 0,
                           batch_first=True,
                           bidirectional=True)  # 양방향 LSTM
        
        # 배치 정규화 레이어
        self.batch_norm = nn.BatchNorm1d(hidden_dim * 2)  # 양방향이므로 hidden_dim * 2
        
        # 드롭아웃 레이어
        self.dropout1 = nn.Dropout(drop_prob)
        self.dropout2 = nn.Dropout(drop_prob * 0.5)
        
        # 완전 연결 레이어들 (더 깊은 네트워크)
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)  # 양방향이므로 hidden_dim * 2
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)
        
    def forward(self, x):
        # 임베딩 레이어 통과
        embedded = self.embedding(x)
        
        # 양방향 LSTM 레이어 통과
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # 마지막 hidden state 추출 (양방향이므로 forward와 backward 연결)
        # hidden shape: (n_layers * 2, batch_size, hidden_dim)
        forward_hidden = hidden[-2]  # 마지막 forward layer
        backward_hidden = hidden[-1]  # 마지막 backward layer
        combined_hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        
        # 배치 정규화 적용
        normalized = self.batch_norm(combined_hidden)
        
        # 첫 번째 완전 연결 레이어
        out = self.dropout1(normalized)
        out = F.relu(self.fc1(out))
        
        # 두 번째 완전 연결 레이어
        out = self.dropout2(out)
        out = F.relu(self.fc2(out))
        
        # 출력 레이어
        out = self.fc3(out)
        
        return out

# 2차 모델: GRU 기반 모델
class ImprovedSentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, drop_prob=0.3):
        super(ImprovedSentimentGRU, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # 임베딩 레이어
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 양방향 GRU 레이어
        self.gru = nn.GRU(embedding_dim, 
                         hidden_dim, 
                         n_layers, 
                         dropout=drop_prob if n_layers > 1 else 0,
                         batch_first=True,
                         bidirectional=True)
        
        # 배치 정규화 레이어
        self.batch_norm = nn.BatchNorm1d(hidden_dim * 2)
        
        # 드롭아웃 레이어
        self.dropout1 = nn.Dropout(drop_prob)
        self.dropout2 = nn.Dropout(drop_prob * 0.5)
        
        # 완전 연결 레이어들
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)
        
    def forward(self, x):
        # 임베딩 레이어 통과
        embedded = self.embedding(x)
        
        # 양방향 GRU 레이어 통과
        gru_out, hidden = self.gru(embedded)
        
        # 마지막 hidden state 추출
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        combined_hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        
        # 배치 정규화 적용
        normalized = self.batch_norm(combined_hidden)
        
        # 첫 번째 완전 연결 레이어
        out = self.dropout1(normalized)
        out = F.relu(self.fc1(out))
        
        # 두 번째 완전 연결 레이어
        out = self.dropout2(out)
        out = F.relu(self.fc2(out))
        
        # 출력 레이어
        out = self.fc3(out)
        
        return out

# 개선된 모델 하이퍼파라미터 설정
improved_embedding_dim = 256  # 더 큰 임베딩 차원
improved_hidden_dim = 512     # 더 큰 은닉 차원
improved_n_layers = 3         # 더 많은 레이어
improved_drop_prob = 0.3      # 더 낮은 드롭아웃

# 모델 인스턴스 생성
improved_lstm_model = ImprovedSentimentLSTM(vocab_size, improved_embedding_dim, 
                                           improved_hidden_dim, output_dim, 
                                           improved_n_layers, improved_drop_prob)

improved_gru_model = ImprovedSentimentGRU(vocab_size, improved_embedding_dim, 
                                         improved_hidden_dim, output_dim, 
                                         improved_n_layers, improved_drop_prob)

print("개선된 LSTM 모델:")
print(improved_lstm_model)
print("\n개선된 GRU 모델:")
print(improved_gru_model)


In [ ]:
# 개선된 LSTM 모델 학습 및 평가
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

def train_and_evaluate_model(model, model_name, num_epochs=15, learning_rate=0.001):
    print(f"\n{'='*50}")
    print(f"{model_name} 학습 시작")
    print(f"{'='*50}")
    
    # 손실 함수 및 옵티마이저 정의
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    # 학습률 스케줄러 (성능이 개선되지 않으면 학습률 감소)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)
    
    # 장치 할당
    model.to(device)
    
    # 학습 과정 추적을 위한 리스트
    train_losses = []
    test_accuracies = []
    best_accuracy = 0
    
    # 학습 루프
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping (그래디언트 폭주 방지)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # 평가
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        
        # 학습률 스케줄러 업데이트
        scheduler.step(accuracy)
        
        # 최고 성능 업데이트
        if accuracy > best_accuracy:
            best_accuracy = accuracy
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Accuracy: {accuracy:.2f}%')
    
    print(f"\n{model_name} 학습 완료!")
    print(f"최고 정확도: {best_accuracy:.2f}%")
    
    return train_losses, test_accuracies, best_accuracy

# 1. 개선된 LSTM 모델 학습
lstm_train_losses, lstm_test_accuracies, lstm_best_acc = train_and_evaluate_model(
    improved_lstm_model, "개선된 LSTM 모델", num_epochs=15, learning_rate=0.001
)


In [ ]:
# 2. 개선된 GRU 모델 학습
gru_train_losses, gru_test_accuracies, gru_best_acc = train_and_evaluate_model(
    improved_gru_model, "개선된 GRU 모델", num_epochs=15, learning_rate=0.001
)


In [ ]:
# 3. 성능 비교 및 시각화
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 기존 모델의 최종 성능 (앞서 학습된 결과에서)
original_best_acc = max(test_accuracies) if test_accuracies else 15.64

print("\n" + "="*60)
print("모델 성능 비교 결과")
print("="*60)
print(f"기존 LSTM 모델 최고 정확도: {original_best_acc:.2f}%")
print(f"개선된 LSTM 모델 최고 정확도: {lstm_best_acc:.2f}%")
print(f"개선된 GRU 모델 최고 정확도: {gru_best_acc:.2f}%")
print("-"*60)
print(f"LSTM 모델 성능 향상: {lstm_best_acc - original_best_acc:.2f}%p")
print(f"GRU 모델 성능 향상: {gru_best_acc - original_best_acc:.2f}%p")
print("="*60)

# 성능 비교 차트
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('학습 손실 비교', '정확도 비교', '최종 성능 비교', '모델별 정확도 추이'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# 1. 학습 손실 비교
if 'train_losses' in locals():  # 기존 모델 손실이 있다면
    fig.add_trace(
        go.Scatter(y=train_losses, mode='lines', name='기존 LSTM', line=dict(color='red')),
        row=1, col=1
    )

fig.add_trace(
    go.Scatter(y=lstm_train_losses, mode='lines', name='개선된 LSTM', line=dict(color='blue')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=gru_train_losses, mode='lines', name='개선된 GRU', line=dict(color='green')),
    row=1, col=1
)

# 2. 정확도 비교
if 'test_accuracies' in locals():  # 기존 모델 정확도가 있다면
    fig.add_trace(
        go.Scatter(y=test_accuracies, mode='lines', name='기존 LSTM', line=dict(color='red')),
        row=1, col=2
    )

fig.add_trace(
    go.Scatter(y=lstm_test_accuracies, mode='lines', name='개선된 LSTM', line=dict(color='blue')),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(y=gru_test_accuracies, mode='lines', name='개선된 GRU', line=dict(color='green')),
    row=1, col=2
)

# 3. 최종 성능 바 차트
models = ['기존 LSTM', '개선된 LSTM', '개선된 GRU']
accuracies = [original_best_acc, lstm_best_acc, gru_best_acc]
colors = ['red', 'blue', 'green']

fig.add_trace(
    go.Bar(x=models, y=accuracies, marker_color=colors, name='최고 정확도'),
    row=2, col=1
)

# 4. 개선 효과 바 차트
improvements = [0, lstm_best_acc - original_best_acc, gru_best_acc - original_best_acc]
fig.add_trace(
    go.Bar(x=models, y=improvements, marker_color=colors, name='성능 향상'),
    row=2, col=2
)

fig.update_layout(
    title='한국어 감정 분류 모델 성능 개선 결과',
    height=800,
    showlegend=True
)

fig.show()

# 가장 성능이 좋은 모델 선택
best_models = {'LSTM': (improved_lstm_model, lstm_best_acc), 
               'GRU': (improved_gru_model, gru_best_acc)}

best_model_name = max(best_models, key=lambda x: best_models[x][1])
best_model, best_acc = best_models[best_model_name]

print(f"\n최고 성능 모델: {best_model_name} (정확도: {best_acc:.2f}%)")


In [ ]:
# 4. 개선된 모델로 새로운 문장 감성 예측 테스트
def predict_emotion_improved(text, model):
    model.eval()
    
    # 1. 텍스트 전처리
    cleaned_text = preprocess_korean_text(text)
    tokenized = cleaned_text.split()
    
    # 2. 정수 인코딩 (사전에 없는 단어는 <unk>)
    encoded = [word_to_idx.get(word, 0) for word in tokenized]
    
    # 3. 패딩
    padded = np.array([encoded[:max_len] + [0]*(max_len - len(encoded)) if len(encoded) < max_len else encoded[:max_len]])
    
    # 4. 텐서 변환 및 장치 할당
    input_tensor = torch.LongTensor(padded).to(device)
    
    # 5. 예측 및 확률 계산
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)
        confidence, prediction = torch.max(probabilities, 1)
    
    # 감정 레이블 매핑
    emotion_labels = ['행복', '놀람', '분노', '공포', '혐오', '슬픔', '중립']
    predicted_emotion = emotion_labels[prediction.item()]
    confidence_score = confidence.item()
    
    return predicted_emotion, confidence_score

# 다양한 테스트 문장들
test_sentences = [
    "오늘 정말 기분이 좋아요! 새로운 일이 생겨서 너무 신나요.",
    "이런 상황이 정말 화가 나네요. 도저히 참을 수 없어요.",
    "무서운 영화를 보고 밤에 잠을 못 자겠어요.",
    "친구가 갑자기 나타나서 정말 놀랐어요!",
    "그 사람 행동이 정말 역겨워요.",
    "사랑하는 사람을 잃어서 너무 슬퍼요.",
    "그냥 평범한 하루였어요."
]

expected_emotions = ['행복', '분노', '공포', '놀람', '혐오', '슬픔', '중립']

print(f"\n{'='*80}")
print(f"최고 성능 모델({best_model_name})을 사용한 감정 예측 테스트")
print(f"{'='*80}")

correct_predictions = 0
for i, (sentence, expected) in enumerate(zip(test_sentences, expected_emotions)):
    predicted, confidence = predict_emotion_improved(sentence, best_model)
    is_correct = predicted == expected
    if is_correct:
        correct_predictions += 1
    
    print(f"{i+1}. 문장: '{sentence}'")
    print(f"   예상 감정: {expected} | 예측 감정: {predicted} | 신뢰도: {confidence:.3f} | {'✓' if is_correct else '✗'}")
    print()

accuracy_on_test = correct_predictions / len(test_sentences) * 100
print(f"테스트 문장 정확도: {accuracy_on_test:.1f}% ({correct_predictions}/{len(test_sentences)})")
print(f"{'='*80}")

# 5. 개선사항 요약
print(f"\n{'='*80}")
print(f"모델 개선사항 요약")
print(f"{'='*80}")
print("1. 양방향 LSTM/GRU 사용 - 문맥의 양방향 정보 활용")
print("2. 더 큰 임베딩 차원 (128 → 256) - 더 풍부한 단어 표현")
print("3. 더 큰 은닉 차원 (256 → 512) - 더 복잡한 패턴 학습")
print("4. 더 많은 레이어 (2 → 3) - 더 깊은 네트워크")
print("5. 배치 정규화 추가 - 안정적인 학습")
print("6. 다층 완전연결 레이어 - 더 복잡한 분류")
print("7. 학습률 스케줄링 - 적응적 학습률")
print("8. 그래디언트 클리핑 - 안정적인 학습")
print("9. 가중치 감쇠 - 과적합 방지")
print("10. 더 긴 학습 시간 (10 → 15 에포크)")
print(f"{'='*80}")


In [ ]:
# 모델 저장 및 제출
import torch
import os

# 모델 저장 디렉토리 생성
save_dir = '../../models/emotion_classification/'
os.makedirs(save_dir, exist_ok=True)

# 모델 상태 저장
model_path = os.path.join(save_dir, 'emotion_classification_model.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
    'loss': train_losses[-1],
    'word_to_idx': word_to_idx,
    'max_len': max_len,
    'embedding_dim': embedding_dim,
    'hidden_dim': hidden_dim,
    'n_layers': n_layers,
    'vocab_size': vocab_size,
    'device': device.type
}, model_path)

print(f"모델이 '{model_path}' 파일로 저장되었습니다.")
